# CHURRO-3B HTR Baseline

> **GPU required:** L4 (24 GB) or A100 (40 GB). **Recommended: L4.**
> In Colab: *Runtime → Change runtime type → L4.*
> This model needs ~20 GB VRAM and will OOM on T4 (16 GB).

This notebook transcribes evaluation subset pages using
[CHURRO-3B](https://huggingface.co/stanford-oval/churro-3B), a **3-billion-parameter**
vision-language model purpose-built for **Handwritten Text Recognition (HTR)** on
historical manuscripts.

### What makes CHURRO-3B interesting

- **Architecture**: Fine-tuned from **Qwen2.5-VL-3B-Instruct**, inheriting its NaViT
  vision encoder and Qwen2.5 language backbone.
- **Training data**: 99K historical manuscript page images with paired transcriptions,
  covering a wide range of scripts, centuries, and writing styles.
- **Design goal**: Specifically optimised for HTR rather than general OCR, making it one
  of the first open-weight models targeted at paleography.

| Detail | Value |
|--------|-------|
| HuggingFace ID | `stanford-oval/churro-3B` |
| Parameters | ~3 B |
| Base model | Qwen2.5-VL-3B-Instruct |
| VRAM required | ~20 GB (needs L4 24 GB or A100 40 GB) |
| dtype | `bfloat16` |

## 1. Setup & Data Access

We mount Google Drive so the notebook can read the evaluation images that have been
pre-uploaded to `MyDrive/paleo-ocr/subset_images/`. Four path variables control where
inputs are read from and where outputs are written:

- **`DRIVE_BASE`** -- root of the paleo-ocr project on Drive.
- **`IMAGES_DIR`** -- folder containing the manuscript page images.
- **`MANIFEST_PATH`** -- JSON file listing every page to transcribe (page ID,
  canonical file name, document metadata).
- **`OUTPUT_DIR`** -- where raw transcription `.txt` files are saved (one per page).

In [ ]:
# Mount Google Drive (images pre-uploaded)
import os
from google.colab import drive
drive.mount('/content/drive')

# Cache HuggingFace models on local storage (wiped when session ends, saves Drive space)
os.environ['HF_HOME'] = '/content/.hf_cache'

# Authenticate with HuggingFace for faster downloads (optional)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except (ImportError, userdata.SecretNotFoundError):
    print('HF_TOKEN not found — downloads will be unauthenticated (slower)')

# Configure paths
DRIVE_BASE = '/content/drive/MyDrive/paleo-ocr'
IMAGES_DIR = f'{DRIVE_BASE}/subset_images'
MANIFEST_PATH = f'{DRIVE_BASE}/evaluation_subset.json'
OUTPUT_DIR = f'{DRIVE_BASE}/data/results/churro_3b/raw'

## 2. Install Dependencies

| Package | Purpose |
|---------|--------|
| `transformers` | Model and processor classes (`AutoModelForImageTextToText`, `AutoProcessor`) |
| `torch` | Tensor operations, GPU inference |
| `accelerate` | `device_map='auto'` support for automatic GPU placement |
| `pillow` | Image loading and conversion |
| `qwen-vl-utils` | `process_vision_info` helper required by the Qwen2.5-VL chat template |

In [ ]:
!pip install -q transformers torch accelerate pillow qwen-vl-utils

## 3. Load Manifest & Prepare Output

The manifest is a JSON list where each entry contains at minimum:

```json
{"page_id": "doc001_p01", "canonical_name": "doc001_p01.jpg", ...}
```

We iterate over every entry later.  The output directory is created if it does not
already exist.  **Resumability**: any page whose output file already exists and is
non-empty will be skipped during transcription, so the notebook can be re-run after
interruptions without re-processing finished pages.

In [ ]:
import json
import os
import re
from pathlib import Path
from PIL import Image
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from qwen_vl_utils import process_vision_info

# Load manifest
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
print(f'Loaded {len(manifest)} pages')

# >>> TRIAL RUN: process only CODEA-0143 (2 pages) — remove this line for full evaluation
#manifest = [e for e in manifest if e['doc_id'] == 'CODEA-0143']
#print(f'  Filtered to {len(manifest)} pages (trial run: CODEA-0143 only)')

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Load Model

CHURRO-3B is loaded with `AutoModelForImageTextToText` -- the recommended Auto class
for vision-language models in transformers 5.x, matching the
[official CHURRO inference script](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py).

Key options (all aligned with the official reference):

- **`dtype=torch.bfloat16`** -- matches the stored weight format (`bfloat16`)
  and the official script's dtype selection
  ([source](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L77)).
  BF16 is preferred over FP16 on Ampere+ GPUs (A100) as it avoids potential overflow
  in the exponent range. (`torch_dtype` was renamed to `dtype` in transformers 5.x.)
- **`device_map='auto'`** -- lets `accelerate` place layers on the GPU automatically.
- **`trust_remote_code=True`** -- required because the model repo ships custom
  modelling code.

The attention backend is **auto-detected** by transformers -- it will use Flash
Attention 2 on Ampere+ GPUs (A100), SDPA on Turing (T4/L4), or eager as fallback.
No need to hardcode `attn_implementation`.

### Image resolution (`min_pixels` / `max_pixels`)

The official script explicitly sets pixel bounds in the processor
([source](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L85-L86)):

```python
MIN_PIXELS = 512 * 28 * 28    # = 401,408  (~0.4 MP)
MAX_PIXELS = 5120 * 28 * 28   # = 4,014,080 (~4 MP)
```

**Note:** CHURRO's custom processor (`trust_remote_code=True`) does not forward
`min_pixels` / `max_pixels` kwargs from `AutoProcessor.from_pretrained()` to the
image processor. We set them directly on `processor.image_processor` after loading.

For our dataset:

- **Toledo** images (1.8–2.6 MP) pass through at full resolution.
- **CODEA** images (8–12 MP) are downsampled ~3× to fit within the 4 MP budget.

In [ ]:
MODEL_ID = 'stanford-oval/churro-3B'

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
processor.image_processor.min_pixels = 512 * 28 * 28
processor.image_processor.max_pixels = 5120 * 28 * 28

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
print('Model loaded successfully')
print(f'max_pixels: {processor.image_processor.max_pixels}')

## 5. Transcribe Pages

**Prompt strategy** (matches the
[official CHURRO inference script](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py)):

- A **system message** --
  *"Transcribe the entiretly of this historical documents to XML format."* --
  is the default prompt CHURRO was fine-tuned with
  ([source](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L97)).
  The typo ("entiretly") is intentional -- it matches the training data exactly.
- The **user message** contains **only the image**, with no additional text
  ([source](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L98-L100)).

**Image preprocessing guard**: Before tokenization, images are resized so the longest
side is ≤ 2500 px, matching CHURRO's built-in preprocessing
([source](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L89-L95)).
This prevents excessively large images from producing too many vision tokens.

**Generation parameters** (all from the official defaults):

| Parameter | Value | Rationale | Source |
|-----------|-------|-----------|--------|
| `max_new_tokens` | `20,000` | CHURRO outputs verbose XML; 2048 may truncate long pages | [default arg](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L23) |
| `temperature` | `0.6` | Avoids degenerate repetition loops in long transcriptions | [default arg](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L25) |
| `do_sample` | `True` | Required when temperature > 0 | [line 106](https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py#L106) |

**Context manager**: `torch.inference_mode()` is used instead of `torch.no_grad()`
for slightly better performance (disables more autograd tracking)
([PyTorch docs](https://pytorch.org/docs/stable/generated/torch.inference_mode.html)).

**Post-processing**: CHURRO-3B wraps output in XML tags (as requested by the system
prompt). We strip all XML tags with a regex before saving plain text.

In [ ]:
# Official CHURRO prompt and preprocessing constants
# Source: https://github.com/stanford-oval/Churro/blob/main/churro_transformers_infer.py
SYSTEM_MESSAGE = 'Transcribe the entiretly of this historical documents to XML format.'
MAX_IMAGE_DIM = 2500

for i, entry in enumerate(manifest):
    page_id = entry['page_id']
    out_path = Path(OUTPUT_DIR) / f'{page_id}_raw.txt'

    # Skip already-processed pages (resumability)
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f'[{i+1}/{len(manifest)}] {page_id} - SKIP')
        continue

    img_path = Path(IMAGES_DIR) / entry['canonical_name']
    if not img_path.exists():
        print(f'[{i+1}/{len(manifest)}] {page_id} - IMAGE NOT FOUND')
        continue

    try:
        image = Image.open(img_path).convert('RGB')
        w, h = image.size
        print(f'  Image: {w}x{h} ({w*h/1e6:.1f} MP)')

        # Match CHURRO's preprocessing guard (max 2500px longest side)
        if max(w, h) > MAX_IMAGE_DIM:
            scale = MAX_IMAGE_DIM / max(w, h)
            image = image.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
            print(f'  Resized to: {image.size[0]}x{image.size[1]}')

        # Build Qwen2.5-VL chat messages (official structure: system + image-only user)
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_MESSAGE}]},
            {"role": "user", "content": [{"type": "image", "image": image}]},
        ]

        text_input = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, _ = process_vision_info(messages)
        inputs = processor(
            text=[text_input], images=image_inputs,
            return_tensors='pt', padding=True
        ).to(model.device)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs, max_new_tokens=20_000, do_sample=True, temperature=0.6
            )

        generated = output_ids[:, inputs.input_ids.shape[1]:]
        text = processor.batch_decode(generated, skip_special_tokens=True)[0]

        # Strip XML wrapper tags (model outputs XML as requested by system prompt)
        text = re.sub(r'<[^>]+>', '', text).strip()

        out_path.write_text(text, encoding='utf-8')
        print(f'[{i+1}/{len(manifest)}] {page_id} - {len(text)} chars')

    except Exception as e:
        print(f'[{i+1}/{len(manifest)}] {page_id} - ERROR: {e}')

    # Free cached GPU memory between pages to prevent fragmentation
    torch.cuda.empty_cache()

print('\nDone!')

## 6. Export Results

All raw transcription files are compressed into a single ZIP archive and offered for
download via `google.colab.files`.  The archive mirrors the flat structure of
`OUTPUT_DIR` (`{page_id}_raw.txt` per page), which is the format expected by the
downstream CER/WER evaluation scripts.

In [ ]:
import shutil
shutil.make_archive('/content/churro_3b_results', 'zip', OUTPUT_DIR)

from google.colab import files
files.download('/content/churro_3b_results.zip')